## Mostafa Zamaniturk

---
## Clean up

No Vector Search resources to clean up — the embeddings are stored in a Delta table (`main.default.ultrafeedback_embeddings`) which persists across assignments at no additional cost.


---
## 1. Why evaluate agents?

An agent that *seems* to work in a few demos might fail on edge cases, hallucinate tool results, or give confidently wrong answers. **Evaluation** is how you find out before your users do.

MLflow 3 provides three types of judges that leverage LLMs that we will use in this assignment:

| Judge type | What it does | When to use |
|-----------|-------------|-------------|
| **Built-in** (e.g., `Correctness`, `Safety`) | Pre-configured scorers with standard rubrics | Quick baseline — does the agent give correct, safe answers? |
| **Guidelines** | You write natural-language rules; the judge checks compliance | Domain-specific quality bars (e.g., "must cite the data source") |
| **Custom** (`make_judge()`) | You write the full prompt template | Full control — your own rubric, scoring, and output format |

In this assignment you'll use all three on your UltraFeedback Expert agent and compare how they rate the same outputs.

---
## 2. Install dependencies

In [0]:
%pip install --upgrade "mlflow>=3.9" databricks-langchain unitycatalog-ai[databricks] numpy databricks-agents backoff databricks-openai
dbutils.library.restartPython()

In [0]:
%restart_python

In [0]:
dbutils.library.restartPython()

---
## 3. Verify embeddings table

The embeddings table (`main.default.ultrafeedback_embeddings`) was created in Assignment 3. Verify it exists and has the expected data.


In [0]:
# Verify the embeddings table from Assignment 3 exists
emb_df = spark.table("main.default.ultrafeedback_embeddings")
print(f"Embeddings table has {emb_df.count()} rows.")
print(f"Columns: {emb_df.columns}")
display(emb_df.select("id", "instruction", "source", "embedding").limit(5))


## 4. Load the agent from Assignment 4

Load the agent you created in Assignment 4 from the `agent.py` file. Ensure `agent.py` is in the same directory as this notebook (or on the Python path). The cell sets the shared experiment and creates an `agent_executor`-compatible wrapper so the rest of the evaluation notebook works unchanged.


In [0]:
ls

In [0]:
import os
print(os.getcwd())

In [0]:
%pip install "databricks-vectorsearch<0.74" --upgrade --quiet

In [0]:
%restart_python

In [0]:
import mlflow

# Shared experiment across Assignments 4, 5, 6
EXPERIMENT_NAME = "/Users/" + spark.sql("SELECT current_user()").first()[0] + "/aai510_ultrafeedback_expert"
mlflow.set_experiment(EXPERIMENT_NAME)

# Load the agent from Assignment 4's agent.py (same directory or on Python path)
import agent
AGENT = agent.ToolCallingAgent(llm_endpoint=agent.LLM_ENDPOINT_NAME, tools=agent.TOOL_INFOS)


def _response_to_text(resp):
    """Extract final answer text from ResponsesAgentResponse."""
    text_parts = []
    for item in resp.output:
        c = getattr(item, "content", None)
        if c is None:
            continue
        for part in (c if isinstance(c, list) else [c]):
            if isinstance(part, dict) and "text" in part:
                text_parts.append(part["text"])
            elif hasattr(part, "text"):
                text_parts.append(part.text)
    return "\n".join(text_parts) if text_parts else str(resp.output)[:500]


# Wrapper so evals can use agent_executor.invoke(inputs) -> {"output": "..."}
class AgentExecutorWrapper:
    def invoke(self, inputs):
        resp = AGENT.predict({
            "input": [{"role": "user", "content": inputs["input"]}],
            "custom_inputs": {"session_id": "aai510-eval"},
        })
        return {"output": _response_to_text(resp)}


agent_executor = AgentExecutorWrapper()

# Quick test
resp = agent_executor.invoke({"input": "How many rows come from evol_instruct?"})
print("Agent test:", resp["output"][:200])

---
## 5. Create an evaluation dataset

An evaluation dataset is a set of questions (inputs) with optional expected answers (expectations). You need enough variety to test different agent capabilities.

We'll create the dataset in two ways:
1. **Manually** — you write questions that specifically target your tools
2. **Synthetically** — use an LLM to generate additional questions

The combined list is written to a **UC-backed evaluation dataset** (`main.default.ultrafeedback_expert_eval`) via `create_dataset` / `merge_records`, and that dataset is passed to `mlflow.genai.evaluate()` in Section 9.

**Docs:** [Build evaluation datasets](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/build-eval-dataset) · [evaluate-improve-genai-app](https://docs.databricks.com/aws/en/notebooks/source/mlflow3/evaluate-improve-genai-app.html)

### Your task

Add 5 more questions, where at least 3 test your custom UC functions.


In [0]:
# ---- Manual evaluation questions ----
# Each row = one trace. The 'inputs' column must be a dict whose keys match predict_fn's parameter names.
# Use a single parameter name (e.g. 'question') so MLflow calls predict_fn(question="...") once per row — not as one multi-turn session.
manual_eval_data = [
    {"inputs": {"question": "What model pairs are in the UltraFeedback dataset? Show me the top few."}},
    {"inputs": {"question": "How did gpt-4 compare to llama-2-7b-chat in the dataset?"}},
    {"inputs": {"question": "Analyze the complexity of: What is 2+2?"}},
    {"inputs": {"question": "Find instructions in the dataset similar to 'python programming'."}},
    {"inputs": {"question": "What is the capital of France?"}},
    {"inputs": {"question": "Compare gpt-3.5-turbo vs alpaca-7b and gpt-4 vs wizardlm-7b. Which pair has more comparisons?"}},
    {"inputs": {"question": "What models are involved in the dataset?"}},
    # ---- YOUR 5 QUESTIONS BELOW ----
    {"inputs": {"question": "What is quality assurance"}},
    {"inputs": {"question": "What do special inspector do"}},
    {"inputs": {"question": "What factors a professional insoection report have?"}},
    {"inputs": {"question": "Give me top three models for quality Assurance purposes."}},
    {"inputs": {"question": "Give me top three sources for quality Assurance purposes."}},
    {"inputs": {"question": "What pairs are compatible for quality Assurance purposes."}},
]

print(f"Manual eval dataset: {len(manual_eval_data)} questions")

In [0]:
# Combine into the final evaluation dataset
# (If synthetic generation above didn't work, that's OK — the manual set is sufficient)
eval_data = manual_eval_data
print(f"Total evaluation dataset: {len(eval_data)} questions")

# Create a UC-backed evaluation dataset and merge records (Databricks pattern:
# https://docs.databricks.com/aws/en/notebooks/source/mlflow3/evaluate-improve-genai-app.html)
uc_schema = "main.default"
evaluation_dataset_table_name = "ultrafeedback_expert_eval"
full_table_name = f"{uc_schema}.{evaluation_dataset_table_name}"
if not spark.catalog.tableExists(full_table_name):
    eval_dataset = mlflow.genai.datasets.create_dataset(name=full_table_name)
else:
    eval_dataset = mlflow.genai.datasets.get_dataset(name=full_table_name)
eval_dataset = eval_dataset.merge_records(eval_data)
# Re-running this cell will append records again; for a fresh dataset, use a new table name or drop the table.
print(f"Evaluation dataset '{full_table_name}' ready ({len(eval_data)} records). Pass eval_dataset to mlflow.genai.evaluate().")

---
## 6. Define judges and predict function

Below we define the **predict function** (wraps your agent) and the **built-in judge** `RelevanceToQuery`, which checks whether the response actually addresses the user's question. All judges are run together in **one** evaluate job in Section 9.

**Docs:** [Built-in scorers](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/judges)

In [0]:
from mlflow.genai.scorers import RelevanceToQuery

# Predict function: one parameter per row so each row = one trace (not one multi-turn conversation).
# Parameter name must match the key in each row's inputs dict (e.g. {"inputs": {"question": "..."}}).
def predict_fn(question: str) -> str:
    """Run the agent on a single question and return the output string."""
    response = agent_executor.invoke({"input": question})
    return response["output"]

# Built-in judge: we'll run it together with all other judges in one evaluate job (Section 9)
relevance_judge = RelevanceToQuery()

---
## 7. Guidelines judges

**Guidelines judges** check whether the response follows specific rules you define in natural language, and will always create a pass/fail judge. Define both judges below; they are included in the single evaluate job in Section 9.

In [0]:
from mlflow.genai.scorers import Guidelines

# Define domain-specific guidelines for the UltraFeedback Expert (run in single evaluate job in Section 9)
tool_citation_judge = Guidelines(
    name="tool_citation",
    guidelines="The response must clearly state which tool or data source was used "
               "to generate the answer. If no tool was needed, the response should "
               "acknowledge that. Vague answers that don't explain their source fail."
)

data_grounding_judge = Guidelines(
    name="data_grounding",
    guidelines="When the question is about the UltraFeedback dataset, the response "
               "must include specific numbers or examples from the actual data "
               "(e.g., row counts, source names, sample instructions). Generic or "
               "made-up statistics fail this criterion."
)

---
## 8. Custom judge — example and your own

A **custom judge** gives you full control over the evaluation prompt via `make_judge()`. You define the rubric, scoring criteria, and output format.

**Example below:** A judge that evaluates whether the agent used its tools appropriately. Notice it explicitly sets **model** (the LLM used to run the judge) and **feedback_value_type** (the type of score returned: `bool`, `int`, `float`, `str`, or e.g. `Literal["good", "neutral", "bad"]`).

**Your task:** In the next cell you will create your **own** custom judge with `make_judge()`. You must explicitly specify: 

- **instructions** (prompt with at least one of `{{ inputs }}`, `{{ outputs }}`, `{{ expectations }}`), 
- **model** (e.g. `"databricks:/databricks-gpt-oss-120b"`), 
- **feedback_value_type** (e.g. `bool`, `int`, or `Literal["good", "neutral", "bad"]`).

The cell then **registers** the scorer to the current experiment (`.register()`) so it is versioned and can be loaded in Assignment 6 with `get_scorer()`. Use the commented-out judge as an example, but do not re-use this judge.

**Docs:** [Custom judges](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge) · [make_judge](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/make-judge/)

In [0]:
from mlflow.genai.judges import make_judge

# Use this judge as an example, do not use this judge:

# Create a custom judge that evaluates tool usage appropriateness
my_custom_judge = make_judge(
    name="tool_usage_quality",
    instructions="""You are evaluating an AI agent's ability to use tools appropriately.

The agent has access to these tools (from Assignments 3–4):
- all_model_combinations: list model pairs and comparison data from the dataset
- analyze_instruction: instruction complexity metrics (word count, sentence count, etc.)
- compare_models: compare two models (e.g. win counts, average chosen/rejected ratings)
- search_similar_instructions: search the embeddings table for similar instructions by keyword
- main__default__find_best_technical_doc_models: find best models for technical documentation tasks
- main__default__get_best_model_source_pairs: find top-performing model-source combinations
- main__default__recommend_reasoning_agent_route: recommend model-source pairs for reasoning agents
- You.com MCP (if configured): live web search

User question: {{inputs}}
Agent response: {{outputs}}

Evaluate the agent's tool usage:
1. Did the agent call the appropriate tool(s) for this question?
2. If the question didn't need a tool, did the agent correctly avoid using one?
3. Did the agent use the tool results correctly in its response?

Return YES if tool usage was appropriate, NO if it was not.
Explain your reasoning.""",
    model="databricks:/databricks-gpt-oss-120b",  # Judge model: explicitly set, use your choice
    feedback_value_type=bool,  # Output: True/False (YES/NO)
)

'''
# Create your own custom judge: set name, instructions (or judge_prompt), model, and feedback_value_type
my_custom_judge = make_judge(
    name="my_custom_judge",  # Your judge name
    instructions="""Your instructions here. Use {{ inputs }} and {{ outputs }}.
Explain what you are evaluating and what format to return.""",
    model="databricks:/databricks-gpt-oss-120b",  # Explicit: judge model
    feedback_value_type=bool,  # Explicit: e.g. bool | int | float | str | Literal["good", "neutral", "bad"]
)
my_custom_judge = my_custom_judge.register()  # Register so Assignment 6 can load with get_scorer()

print("My custom judge defined.")
'''

# Register the scorer to the current experiment for versioning; Assignment 6 will load it with get_scorer()
try:
    my_custom_judge.register()  # if you changed the judge name, update my_custom_judge
    print("Custom judge 'tool_usage_quality' registered successfully.")
except ValueError as e:
    if "already been registered" in str(e):
        print("Custom judge 'tool_usage_quality' already registered (from previous run). Using existing registration.")
    else:
        raise


In [0]:
from mlflow.genai.judges import make_judge

# Create a custom judge that evaluates answer correctness and model/source compatibility
inspector_judge = make_judge(
    name="model_source_correctness_judge",
    instructions="""You are evaluating an AI agent's performance on the UltraFeedback dataset. Your task is to judge whether the agent provided a correct answer and identified the exact right model-source combinations.

User question: {{inputs}}
Agent response: {{outputs}}

Evaluate the agent's response against these three core criteria:
1. General Correctness: Did the agent answer the user's question accurately without introducing hallucinations or factual errors?
2. Model & Source Selection: Did the agent extract and present the proper models and data sources corresponding to the query?
3. Domain Task Compatibility: If the user's question was about technical documentation or reasoning tasks, did the agent successfully identify the compatible, top-performing model-source pairs meant for those specific workflows?

Return YES if the agent's answer is completely correct, lists the proper models/sources, and correctly handles technical documentation or reasoning compatibility.
Return NO if the answer is factually wrong, misses the correct models/sources, or misidentifies the compatible pairs for technical or reasoning tasks.

Explain your reasoning clearly by highlighting where the agent succeeded or failed.""",
    model="databricks:/databricks-gpt-oss-120b",  
    feedback_value_type=bool,  # Outputs True for YES, False for NO
)

# Register the scorer to the current experiment for versioning and Assignment 6 loading
inspector_judge.register()

print("Revised correctness and task-compatibility judge defined and registered.")

---
## 9. Run evaluation and analyze

Run **one** evaluation job with all judges (built-in, both guidelines, and your custom judge), using the UC-backed **eval_dataset** from Section 5. Then compare how they rate the same outputs.

In [0]:
import mlflow
# Single evaluate job: all judges run together; data=eval_dataset uses the UC-backed dataset from Section 5
print("Running evaluation with all judges...")
all_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=[
        relevance_judge,
        tool_citation_judge,
        data_grounding_judge,
        #my_custom_judge,  # Your custom judge from the cell above; if you changed the judge name, updatethis
        inspector_judge, # custom judge
        
    ]
)

print("\nEvaluation complete. See the MLflow UI for the full results table.")

### Screenshot: Evaluation results

Before answering the analysis questions below, capture a screenshot of your evaluation run from the **MLflow Experiments UI** and save it for your submission.

**Steps:**

1. In the **left pane** of the Databricks workspace, open the **Experiments** tab.
2. Under your experiment, open the run that contains the evaluation (or click the link in the cell output above).
3. Go to **Evaluation Runs** (or the **Evaluations** tab for that run).
4. If **Group by Session** is selected, **unselect it** so that each trace from the evaluation appears as its own row (one row per question).
5. Capture a screenshot that shows **at least 12 traces**: the 7 provided questions plus your 5 student-written questions. The table should show the assessment columns (e.g. data_grounding, Relevance, tool_usage_quality) and pass/fail or true/false for each trace.

Name the file something like `eval_results.png` and add it to your root folder with your notebook so it displays in the notebook. If you cannot get that to work, submit it as a separate file in the final submission

### Your analysis 

**Do not use AI for this portion, answer each question in 2-3 sentences**

Review the evaluation results in the MLflow UI (click the link in the cell output above, then go to the **Evaluations** tab). Answer these questions:

**9a. Which judge was strictest? Which was most lenient?**

*Look at the pass/fail rates across all judges. Which one failed the most responses?*

The most stringent evaluator was the custom model_source_correctness_judge, which had a low acceptance rate of 23%. This result indicates significant weaknesses in the agent’s ability to correctly match models to relevant data sources and indicates the need for improved rapid design or tuning of supporting tools.

In contrast, the data_grounding judge was the most lenient, with an acceptance rate of 85%. This suggests that although the agent’s final answers are generally grounded in the context provided, they often do not meet the more rigorous standards of actual accuracy required by the custom correctness criterion. The RelevanceToQuery judge fell in between, with an acceptance rate of 77%.


**9b. Where did the judges disagree?**

*Find specific examples where one judge passed and another failed the same response. Why did they disagree? What does this tell you about evaluation design? Or if this scenario did not happen, why do you think that is the case?*

A clear example of this discrepancy can be seen in the search query: “Find instructions in datasets similar to ‘Python programming.’” In this case, the agent successfully passed the *RelevanceToQuery* assessment but failed the *model_source_correctness_judge*.

This difference highlights an important insight into assessment design: relevance is not the same as correctness. An agent may provide an answer that exactly matches the user’s intent and context of the conversation, but still fail to retrieve the correct technical schemas or model-data pairs. This illustrates why an effective evaluation framework should rely on multiple expert judges rather than a single, overarching metric.

**9c. What did the evaluation reveal about your agent?**

*What are the agent's strengths and weaknesses based on the evaluation? If you were to improve the agent, what would you change first — the tools, the prompt, or the LLM? Was anything missing from your evaluation suite?*


Strengths and Weaknesses: The evaluation showed that the agent is strong at maintaining conversational relevance and overall alignment, as evidenced by high relevance scores. However, its main weakness lies in processing highly technical engineering questions and simple out-of-domain baselines. Interestingly, the agent failed to search for the initial criterion “What is the capital of France?”, while it performed better on specific and complex instructions, suggesting that the notification system is restricting its out-of-domain behavior too aggressively.

First Action Item (LLM): My first step to increase performance will be to migrate and test alternative LLM backends. Evaluating how higher-capability reasoning models interact with this specialized dataset will immediately indicate whether the problem stems from limitations in the base model or the framework itself.

Second Action Item (LLM): My second step will be to refactor the notification system. Instead of relying on long explanations, I will summarize the instructions into concise, direct, and declarative rules. This clearer declaration structure reduces the amount of text and guides the model toward more accurate parameter extraction. 

Last Action Item (Multi-Agent Architecture): Finally, I will move the system from a single, unified agent to a modular, multi-agent orchestration architecture. First, I will create an input supervisor/orchestrator layer that acts as a semantic router to categorize incoming queries based on purpose and complexity (e.g., routing a basic question rather than a very specific engineering request). These categorized queries are then distributed to a fleet of expert worker agents designed for those specific tasks. At the end of the loop, a secondary consensus orchestrator layer evaluates and combines the outputs of the expert workers, ensuring that only the highest quality and most accurate answers are presented to the user.


---
## Lab complete

### Required (Sections 1–9)
- [ ] **Section 3:** Embeddings table verified.
- [ ] **Section 4:** Agent loaded and verified with a test query.
- [ ] **Section 5:** Evaluation dataset created (8+ provided, 5 your own) and UC-backed `eval_dataset` passed to evaluate.
- [ ] **Section 6:** Predict function and built-in judge (`RelevanceToQuery`) defined.
- [ ] **Section 7:** Guidelines judges (`tool_citation`, `data_grounding`) defined.
- [ ] **Section 8:** Custom judge (example and/or your own via `make_judge`) defined.
- [ ] **Section 9:** Single evaluation job ran with all judges; analysis questions answered.
- [ ] **Clean up:** No Vector Search cleanup needed (embeddings persist in Delta table).

**Submit:** Your executed notebook (`.ipynb` with all outputs) and the completed `SUBMISSION_5.md`.

*Next week you'll generate traces, provide human feedback, and use `align()` to improve your judges.*